# SITPCL-FoT + Adaptive Entmax + Evidence Injection (P1 — Stanceosaurus)

**Ba thay đổi so với `sitpcl-fot-sparsemax-with-stanceosaurus.ipynb`, dựa trên phân tích bottleneck thực tế của dữ liệu:**

> ⚠️ **Đính chính (2 lần):**
> 1. Evidence Injection **không phải** "fix lỗi mất dữ liệu / so sánh bất công" — đã kiểm tra lại code thật: cả 3 pipeline cũ đều chỉ đọc `train_conversations.json`, không dùng `title`. Lý do baseline ổn định hơn thực ra nằm ở cách xử lý: baseline gộp cả hội thoại vào 1 chuỗi, để self-attention gốc của BERTweet tự liên kết thông tin (lượt cần phân loại đặt ở đầu chuỗi nên không bị cắt mất); SITPCL mã hoá từng lượt riêng rồi trông cậy GRU+attention tự xây để ghép lại — yếu hơn, lại cộng thêm 2 loss phụ (TPCL+FoT) dễ nhiễu gradient.
> 2. Vì lý do trên, **mặc định của notebook này đã đổi: `use_evidence_injection = False`**. Lúc đầu tôi để mặc định `True`, nhưng điều đó phá vỡ nguyên tắc "cùng input" mà 3 pipeline cũ tuân thủ — nếu thêm `title` mà Macro-F1 thay đổi, ta không biết đó là do kiến trúc mới (entmax/aux-head) hay do có thêm dữ liệu. Mặc định bây giờ **chỉ dùng `train_conversations.json`**, giống hệt 3 pipeline cũ, để so sánh công bằng. Evidence Injection chỉ nên bật (`True`) như MỘT THỬ NGHIỆM RIÊNG, so sánh với chính bản `False` — không so với bảng tham chiếu.

1. **Adaptive Entmax Attention (thay Sparsemax cứng) — đây mới là thay đổi chính, cùng input:** dùng `entmax_bisect` (Peters/Correia/Niculae/Martins, *Adaptively Sparse Transformers*, 2019) với tham số α học được riêng cho từng attention — nội suy mềm giữa softmax (α=1) và sparsemax (α=2) thay vì luôn cắt cứng về 0.
2. **Auxiliary Relevance Head (kiểm định giả thuyết relevance-gate theo kiểu joint, an toàn hơn cascade) — cũng cùng input:** thêm 1 head nhị phân "Other vs Non-Other" học đồng thời với trọng số nhỏ `gamma`, không thay cấu trúc 3 lớp chính.
3. **Evidence Injection (THỬ NGHIỆM RIÊNG, mặc định TẮT, cần dataset thứ 2):** nếu bật, join theo `status_id` với Excel gốc và chèn `title` (tiêu đề fact-check — tín hiệu chưa pipeline nào từng dùng) làm "utterance bằng chứng" (speaker đặc biệt `-1`) vào đầu mỗi hội thoại khi có.

**Yêu cầu Kaggle Input:** chỉ cần **1 dataset bắt buộc** — `Stanceosaurus_reconstructed_dialogues` (train/dev/test_conversations.json), giống hệt 3 pipeline cũ. Dataset `Stanceosaurus-data-main` (Excel) là **tùy chọn**, chỉ cần khi bạn muốn bật thử nghiệm Evidence Injection riêng (Cell 2: `use_evidence_injection = True`).

In [ ]:
# ==============================================================================
# STEP 1: CÀI ĐẶT THƯ VIỆN & TỰ ĐỘNG DÒ TÌM DỮ LIỆU
# ==============================================================================
!pip install -q transformers openpyxl emoji python-box scikit-learn entmax

import os
import glob
import json
import math
import random
from collections import defaultdict
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from sklearn.metrics import classification_report, f1_score, accuracy_score
from tqdm.auto import tqdm
from box import Box
from entmax import entmax_bisect

# Nguồn BẮT BUỘC (duy nhất) — giống hệt 3 pipeline cũ, để kết quả so sánh công bằng (cùng input).
found_json = glob.glob('/kaggle/input/**/train_conversations.json', recursive=True) + \
             glob.glob('./**/train_conversations.json', recursive=True) + \
             glob.glob('D:/KLTN/**/train_conversations.json', recursive=True)
if not found_json:
    raise FileNotFoundError('❌ Không tìm thấy train_conversations.json! Hãy gắn dataset Stanceosaurus_reconstructed_dialogues.')
JSON_DIR = os.path.dirname(found_json[0])
print(f'--> ✅ Dialogue JSON tại: {JSON_DIR}')

# Nguồn TÙY CHỌN — chỉ cần nếu bạn muốn bật thử nghiệm riêng "Evidence Injection" (use_evidence_injection=True ở Cell 2).
# KHÔNG gắn dataset này cũng chạy được bình thường — mặc định evidence injection đang TẮT để giữ đúng
# nguyên tắc "cùng input" với 3 pipeline cũ (BERTweet baseline, SITPCL, SITPCL-FoT+Sparsemax).
found_xlsx = glob.glob('/kaggle/input/**/Train_final.xlsx', recursive=True) + \
             glob.glob('./**/Train_final.xlsx', recursive=True) + \
             glob.glob('D:/KLTN/**/Train_final.xlsx', recursive=True)
XLSX_DIR = os.path.dirname(found_xlsx[0]) if found_xlsx else None
if XLSX_DIR:
    print(f'--> ℹ️ (Tùy chọn) Excel gốc tìm thấy tại: {XLSX_DIR} — có thể bật use_evidence_injection=True để thử nghiệm riêng.')
else:
    print('--> ℹ️ Không tìm thấy Excel gốc (Stanceosaurus-data-main) — bỏ qua, evidence injection sẽ tự tắt. Đây là input mặc định đúng chuẩn, không cần gắn thêm dataset.')

In [ ]:
# ==============================================================================
# STEP 2: CẤU HÌNH SIÊU THAM SỐ
# ==============================================================================
DEBUG_MODE = False  # <-- ĐỔI THÀNH False KHI MUỐN TREO MÁY QUA ĐÊM TRÊN KAGGLE!

config = Box({
    'json_dir': JSON_DIR,
    'xlsx_dir': XLSX_DIR,
    'debug_mode': DEBUG_MODE,
    'debug_samples_per_class': 40,
    'model_name': 'vinai/bertweet-base',
    'num_classes': 3,                     # 0=Supporting, 1=Refuting, 2=Other
    'max_seq_len': 128,
    'batch_size': 16,
    'accum_steps': 2,
    'learning_rate': 2e-5,
    'weight_decay': 0.01,
    'num_epochs': 1 if DEBUG_MODE else 25,  # Trần epoch cao, dừng thật bằng Early Stopping (xem early_stop_patience)
    'early_stop_patience': 4,             # Dừng sớm nếu Dev Macro-F1 không cải thiện sau 4 epoch liên tiếp
    'warmup_ratio': 0.1,
    'gru_hidden': 768,
    'gru_layer': 1,

    # TPCL (giữ nguyên từ SITPCL gốc)
    'alpha': 0.5,
    'tau': 0.1,

    # FoT
    'use_fot': True,
    'beta': 0.5,
    'tau_fot': 0.1,
    'att_temp': 1.0,

    # === THAY ĐỔI 1: ADAPTIVE ENTMAX (thay vì Sparsemax cứng) ===
    'attention_mode': 'entmax',           # 'entmax' | 'sparsemax' | 'softmax' (dùng để ablation so sánh)
    'entmax_alpha_init': 1.5,             # điểm khởi tạo giữa softmax(1.0) và sparsemax(2.0)

    # === THAY ĐỔI 2: EVIDENCE INJECTION — MẶC ĐỊNH TẮT (False) ===
    # Lý do: 3 pipeline cũ (baseline/SITPCL/SITPCL-FoT) đều chỉ dùng train_conversations.json làm
    # input duy nhất. Để so sánh Macro-F1 công bằng (đúng nguyên tắc "cùng input" của các notebook
    # gốc), kết quả chính của notebook này PHẢI chạy với use_evidence_injection=False.
    # Chỉ bật True khi bạn muốn làm MỘT THỬ NGHIỆM RIÊNG, TÁCH BIỆT để trả lời câu hỏi khác:
    # "thêm tín hiệu title có giúp ích không" — khi đó so sánh với chính con số False, KHÔNG so
    # sánh thẳng với 3 con số cũ (vì lúc này input đã khác, không còn công bằng nữa).
    'use_evidence_injection': False,

    # === THAY ĐỔI 3: AUXILIARY RELEVANCE HEAD (joint, không cascade) ===
    'use_aux_relevance_head': True,
    'gamma_aux': 0.3,                     # trọng số loss phụ, nhỏ để không lấn át stance loss chính

    'loss_type': 'focal',
    'focal_gamma': 2.0,
    'use_amp': True,
    'seed': 42,
    'output_dir': './',
    'best_model_path': './best_sitpcl_entmax_evidence_stanceosaurus.pt'
})

# An toàn: nếu lỡ bật use_evidence_injection=True mà không gắn dataset Excel, tự tắt lại thay vì lỗi.
if config.use_evidence_injection and not config.xlsx_dir:
    print('⚠️ use_evidence_injection=True nhưng không tìm thấy Excel gốc — tự động TẮT lại để tránh lỗi.')
    config.use_evidence_injection = False

def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(config.seed)
config.device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(f'--> 🚀 Thiết bị: {config.device} | Max Epochs: {config.num_epochs} (early stop patience={config.early_stop_patience}) | Attention: {config.attention_mode} | Evidence: {config.use_evidence_injection} | Aux-Relevance: {config.use_aux_relevance_head}')

In [ ]:
# ==============================================================================
# STEP 3: NẠP DỮ LIỆU (train_conversations.json — cùng input với 3 pipeline cũ)
# + BỔ SUNG EVIDENCE (title) KHI BẬT use_evidence_injection — mặc định TẮT, xem Cell 2.
# ==============================================================================
EXCEL_COLUMNS = ['number', 'date', 'status_id', 'in_reply_to_id', 'username', 'meta_info',
                  'claim', 'text', 'context', 'title', 'final_annotation', 'final_leaning', 'note_3']

def build_title_lookup(xlsx_filename):
    path = os.path.join(config.xlsx_dir, xlsx_filename)
    df = pd.read_excel(path)
    df.columns = EXCEL_COLUMNS
    lookup = {}
    for _, row in df.iterrows():
        title = str(row['title']).strip()
        if title and title.lower() != 'nan':
            lookup[row['status_id']] = title
    return lookup

def load_dialogue_json(json_filename, title_lookup):
    path = os.path.join(config.json_dir, json_filename)
    with open(path, 'r', encoding='utf-8') as f:
        samples = json.load(f)

    dialogues = []
    n_injected = 0
    for s in samples:
        sentences = list(s['sentences'])
        speakers = list(s['speakers'])
        sid = s.get('status_id')
        title = title_lookup.get(sid) if config.use_evidence_injection else None
        if title:
            sentences = [title] + sentences
            speakers = [-1] + speakers   # -1 = "tiếng nói" riêng của bằng chứng fact-check, không phải người đối thoại
            n_injected += 1
        dialogues.append({
            'target': str(s['target']).strip(),
            'sentences': sentences,
            'speakers': speakers,
            'label': int(s['label'])
        })
    if config.use_evidence_injection:
        print(f'    [{json_filename}] Đã chèn evidence (title) cho {n_injected}/{len(samples)} hội thoại ({n_injected/len(samples)*100:.1f}%)')
    return pd.DataFrame(dialogues)

if config.use_evidence_injection:
    print('[*] use_evidence_injection=True -> build title lookup từ Excel gốc (thử nghiệm RIÊNG, không dùng cho so sánh công bằng với 3 pipeline cũ)...')
    train_titles = build_title_lookup('Train_final.xlsx')
    dev_titles = build_title_lookup('Dev_final.xlsx')
    test_titles = build_title_lookup('Test_final.xlsx')
else:
    print('[*] use_evidence_injection=False -> chỉ dùng train_conversations.json, giống hệt input của 3 pipeline cũ (so sánh công bằng).')
    train_titles, dev_titles, test_titles = {}, {}, {}

print('[*] Đang nạp dialogue JSON...')
train_df = load_dialogue_json('train_conversations.json', train_titles)
dev_df = load_dialogue_json('dev_conversations.json', dev_titles)
test_df = load_dialogue_json('test_conversations.json', test_titles)

if config.debug_mode:
    k = config.debug_samples_per_class
    train_df = train_df.groupby('label', group_keys=False).apply(lambda x: x.sample(n=min(len(x), k), random_state=config.seed)).reset_index(drop=True)
    dev_df = dev_df.groupby('label', group_keys=False).apply(lambda x: x.sample(n=min(len(x), max(k//2, 5)), random_state=config.seed)).reset_index(drop=True)
    test_df = test_df.groupby('label', group_keys=False).apply(lambda x: x.sample(n=min(len(x), max(k//2, 5)), random_state=config.seed)).reset_index(drop=True)
    print(f'⚠️ [DEBUG MODE] Train={len(train_df)}, Dev={len(dev_df)}, Test={len(test_df)}')
else:
    print(f'--> [FULL DATA] Train : {len(train_df):,} | Dev : {len(dev_df):,} | Test : {len(test_df):,}')

label_names = ['Supporting', 'Refuting', 'Other']
dist_df = pd.DataFrame({
    'Train': train_df['label'].value_counts().sort_index(),
    'Dev': dev_df['label'].value_counts().sort_index(),
    'Test': test_df['label'].value_counts().sort_index()
}, index=range(len(label_names)))
dist_df.index = label_names
display(dist_df)

In [ ]:
# ==============================================================================
# STEP 4: PROMPT TOKENIZATION (có nhãn riêng cho utterance bằng chứng)
# ==============================================================================
tokenizer = AutoTokenizer.from_pretrained(config.model_name, use_fast=False)
mask_token = tokenizer.mask_token
mask_token_id = tokenizer.mask_token_id
print(f'--> Tokenizer mask token: "{mask_token}" (ID: {mask_token_id})')

def speaker_phrase(sp):
    return 'the fact-check evidence says' if sp == -1 else f'user {sp} says'

class SITPCLStanceDataset(Dataset):
    def __init__(self, df, tokenizer, max_len=128):
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.data = df.to_dict('records')

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        sentences, speakers, target = item['sentences'], item['speakers'], item['target']

        utterances = []
        for s_idx, sen in enumerate(sentences):
            prompt = f'{speaker_phrase(speakers[s_idx])}: {sen} . Stance towards {target} is {mask_token}'
            enc = self.tokenizer(prompt, max_length=self.max_len, padding='max_length', truncation=True, return_tensors='pt')
            input_ids = enc['input_ids'].squeeze(0)
            att_mask = enc['attention_mask'].squeeze(0)
            mask_indices = (input_ids == mask_token_id).nonzero(as_tuple=True)[0]
            mask_idx = mask_indices[0].item() if len(mask_indices) > 0 else (att_mask.sum().item() - 2)
            utterances.append({'input_ids': input_ids, 'attention_mask': att_mask, 'mask_idx': mask_idx})

        return {
            'utterances': utterances,
            'speakers': speakers,
            'target': target,
            'label': torch.tensor(item['label'], dtype=torch.long)
        }

def sitpcl_collate_fn(batch):
    all_input_ids, all_att_masks, all_mask_indices = [], [], []
    dia_idx, speakers_list, targets_list, labels_list = [], [], [], []

    curr_idx = 0
    for sample in batch:
        n_utts = len(sample['utterances'])
        dia_idx.append((curr_idx, curr_idx + n_utts))
        curr_idx += n_utts
        for utt in sample['utterances']:
            all_input_ids.append(utt['input_ids'])
            all_att_masks.append(utt['attention_mask'])
            all_mask_indices.append(utt['mask_idx'])
        speakers_list.append(sample['speakers'])
        targets_list.append(sample['target'])
        labels_list.append(sample['label'])

    return {
        'input_ids': torch.stack(all_input_ids),
        'attention_mask': torch.stack(all_att_masks),
        'mask_indices': torch.tensor(all_mask_indices, dtype=torch.long),
        'dia_idx': dia_idx,
        'speakers': speakers_list,
        'targets': targets_list,
        'labels': torch.stack(labels_list)
    }

train_dataset = SITPCLStanceDataset(train_df, tokenizer, config.max_seq_len)
dev_dataset = SITPCLStanceDataset(dev_df, tokenizer, config.max_seq_len)
test_dataset = SITPCLStanceDataset(test_df, tokenizer, config.max_seq_len)

train_loader = DataLoader(train_dataset, batch_size=config.batch_size, shuffle=True, collate_fn=sitpcl_collate_fn)
dev_loader = DataLoader(dev_dataset, batch_size=config.batch_size, shuffle=False, collate_fn=sitpcl_collate_fn)
test_loader = DataLoader(test_dataset, batch_size=config.batch_size, shuffle=False, collate_fn=sitpcl_collate_fn)
print('--> ✅ DataLoader đã khởi tạo xong!')

In [ ]:
# ==============================================================================
# STEP 5: KIẾN TRÚC — ADAPTIVE ENTMAX ATTENTION, FOT, TPCL, AUX RELEVANCE HEAD
# ==============================================================================
class AdaptiveAttention(nn.Module):
    """Attention với 3 chế độ: 'entmax' (alpha học được, nội suy mềm giữa softmax/sparsemax),
    'sparsemax' (cắt cứng về 0, Martins & Astudillo 2016), 'softmax' (baseline mềm truyền thống).
    Dựa trên: Correia, Niculae & Martins, "Adaptively Sparse Transformers" (EMNLP 2019)."""
    def __init__(self, hidden_dim, temperature=1.0, mode='entmax', alpha_init=1.5):
        super().__init__()
        self.query_proj = nn.Linear(hidden_dim, hidden_dim)
        self.key_proj = nn.Linear(hidden_dim, hidden_dim)
        self.value_proj = nn.Linear(hidden_dim, hidden_dim)
        self.scale = hidden_dim ** -0.5
        self.temperature = temperature
        self.mode = mode
        if mode == 'entmax':
            # alpha in (1, 2): alpha=1 -> softmax, alpha=2 -> sparsemax. Parametrize qua sigmoid de luon nam trong khoang mo (1,2).
            init_raw = math.log((alpha_init - 1.0) / (2.0 - alpha_init) + 1e-8)
            self.alpha_raw = nn.Parameter(torch.tensor(float(init_raw)))

    def current_alpha(self):
        if self.mode != 'entmax':
            return None
        return 1.0 + torch.sigmoid(self.alpha_raw)

    def forward(self, query, keys, values):
        if query.dim() == 1:
            query = query.unsqueeze(0)
        q = self.query_proj(query)
        k = self.key_proj(keys)
        v = self.value_proj(values)

        attn_scores = (torch.matmul(q, k.transpose(-2, -1)) * self.scale) / self.temperature

        if self.mode == 'entmax':
            alpha = self.current_alpha()
            attn_weights = entmax_bisect(attn_scores, alpha=alpha, dim=-1)
        elif self.mode == 'sparsemax':
            attn_weights = entmax_bisect(attn_scores, alpha=2.0, dim=-1)
        else:
            attn_weights = torch.softmax(attn_scores, dim=-1)

        output = torch.matmul(attn_weights, v)
        out = output.squeeze(0) if output.size(0) == 1 else output
        return out, q.squeeze(0), k

class SSE_FoT(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        hidden_dim = config.gru_hidden
        self.linear_intra = nn.Linear(hidden_dim * 2, hidden_dim)
        self.linear_inter = nn.Linear(hidden_dim, hidden_dim)
        self.attention_intra = AdaptiveAttention(hidden_dim, temperature=config.att_temp, mode=config.attention_mode, alpha_init=config.entmax_alpha_init)
        self.attention_inter = AdaptiveAttention(hidden_dim, temperature=config.att_temp, mode=config.attention_mode, alpha_init=config.entmax_alpha_init)
        self.tau_fot = config.tau_fot
        self.use_fot = config.use_fot

    def forward(self, utterances, speakers):
        device = utterances.device
        V_lst = []
        last_speaker_idx = dict()
        fot_losses = []
        speakers_tensor = torch.tensor(speakers, device=device)

        for i in range(len(speakers)):
            speaker_id = speakers[i]
            if speaker_id not in last_speaker_idx:
                V_lst.append(utterances[i])
            else:
                prev_idx = last_speaker_idx[speaker_id]
                vh_concat = torch.cat((V_lst[prev_idx], utterances[i]), dim=-1)
                q_intra = self.linear_intra(vh_concat)
                c = utterances[:i+1]

                v_intra, q_vec, k_mat = self.attention_intra(q_intra, c, c)

                if self.training and self.use_fot:
                    hist_speakers = speakers_tensor[:i+1]
                    pos_mask = (hist_speakers == speaker_id)
                    pos_mask[i] = False
                    neg_mask = (hist_speakers != speaker_id)
                    if pos_mask.any() and neg_mask.any():
                        q_norm = F.normalize(q_vec.unsqueeze(0), p=2, dim=-1)
                        k_norm = F.normalize(k_mat, p=2, dim=-1)
                        sim = torch.matmul(q_norm, k_norm.T).squeeze(0) / self.tau_fot
                        pos_sim = sim[pos_mask]
                        all_sim = sim[pos_mask | neg_mask]
                        step_fot_loss = - (torch.logsumexp(pos_sim, dim=0) - torch.logsumexp(all_sim, dim=0))
                        fot_losses.append(step_fot_loss)

                q_inter = self.linear_inter(utterances[i])
                k = torch.stack([V_lst[j] for j in range(prev_idx, i)]) if i > prev_idx else utterances[i].unsqueeze(0)
                v_inter, _, _ = self.attention_inter(q_inter, k, k) if len(k) > 0 else (torch.zeros_like(q_inter), None, None)

                V_lst.append(v_intra + v_inter)

            last_speaker_idx[speaker_id] = i

        total_fot_loss = torch.stack(fot_losses).mean() if len(fot_losses) > 0 else torch.tensor(0.0, device=device)
        return torch.stack(V_lst), total_fot_loss

def compute_target_prototypes(H_final, targets):
    tar_to_utts = defaultdict(list)
    for i, t in enumerate(targets):
        tar_to_utts[t].append(H_final[i][-1])
    unique_targets = list(tar_to_utts.keys())
    prototypes = [torch.stack(tar_to_utts[t]).mean(dim=0) for t in unique_targets]
    return torch.stack(prototypes), unique_targets

def target_prototypical_loss(H_final, targets, tau=0.1):
    if len(H_final) <= 1:
        return torch.tensor(0.0, device=H_final[0].device)
    prototypes, unique_targets = compute_target_prototypes(H_final, targets)
    prototypes = F.normalize(prototypes, p=2, dim=-1)
    embeddings = torch.stack([F.normalize(h[-1], p=2, dim=-1) for h in H_final])
    tar_map = {t: idx for idx, t in enumerate(unique_targets)}
    target_labels = torch.tensor([tar_map[t] for t in targets], device=embeddings.device)
    sim_matrix = torch.matmul(embeddings, prototypes.T) / tau
    return F.cross_entropy(sim_matrix, target_labels)

class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, weights=None):
        super().__init__()
        self.gamma, self.weights = gamma, weights

    def forward(self, logits, targets):
        ce_loss = F.cross_entropy(logits, targets, weight=self.weights, reduction='none')
        pt = torch.exp(-ce_loss)
        return (((1.0 - pt) ** self.gamma) * ce_loss).mean()

counts = [sum(train_df['label'] == c) for c in range(config.num_classes)]
class_weights = torch.tensor([1.0 / (max(c, 1) ** 0.5) for c in counts], dtype=torch.float).to(config.device)
class_weights = class_weights / class_weights.sum()

class SITPCL_Entmax_Model(nn.Module):
    def __init__(self, config, class_weights):
        super().__init__()
        self.config = config
        self.backbone = AutoModel.from_pretrained(config.model_name)
        hidden_size = self.backbone.config.hidden_size
        self.gru = nn.GRU(input_size=hidden_size, hidden_size=config.gru_hidden, num_layers=config.gru_layer, batch_first=True)
        self.sse = SSE_FoT(config)
        self.classifier = nn.Sequential(nn.Dropout(0.2), nn.Linear(config.gru_hidden, config.num_classes))
        self.aux_relevance_head = nn.Linear(config.gru_hidden, 2) if config.use_aux_relevance_head else None
        self.focal_criterion = FocalLoss(gamma=config.focal_gamma, weights=class_weights)
        self.aux_criterion = nn.CrossEntropyLoss()

    def forward(self, input_ids, attention_mask, mask_indices, dia_idx, speakers, targets, labels=None):
        outputs = self.backbone(input_ids=input_ids, attention_mask=attention_mask)
        last_hidden = outputs.last_hidden_state
        batch_indices = torch.arange(last_hidden.size(0), device=last_hidden.device)
        mask_reps = last_hidden[batch_indices, mask_indices, :]

        H_final, dialogue_representations, batch_fot_losses = [], [], []
        for d_id, (st, ed) in enumerate(dia_idx):
            d_reps = mask_reps[st:ed]
            o, _ = self.gru(d_reps.unsqueeze(0))
            o = o.squeeze(0)
            v, dia_fot_loss = self.sse(o, speakers[d_id])
            H_final.append(v)
            dialogue_representations.append(v[-1])
            if self.training and dia_fot_loss > 0:
                batch_fot_losses.append(dia_fot_loss)

        stance_features = torch.stack(dialogue_representations)
        logits = self.classifier(stance_features)
        aux_logits = self.aux_relevance_head(stance_features) if self.aux_relevance_head is not None else None

        loss, loss_dict = None, {}
        if labels is not None:
            focal_loss = self.focal_criterion(logits, labels)
            cl_loss = target_prototypical_loss(H_final, targets, tau=self.config.tau)
            fot_loss = torch.stack(batch_fot_losses).mean() if len(batch_fot_losses) > 0 else torch.tensor(0.0, device=logits.device)

            loss = focal_loss + self.config.alpha * cl_loss + self.config.beta * fot_loss
            loss_dict = {'total': loss.item(), 'focal': focal_loss.item(), 'tpcl': cl_loss.item(), 'fot': fot_loss.item()}

            if aux_logits is not None:
                is_other = (labels == 2).long()
                aux_loss = self.aux_criterion(aux_logits, is_other)
                loss = loss + self.config.gamma_aux * aux_loss
                loss_dict['aux_relevance'] = aux_loss.item()

        return loss, logits, aux_logits, loss_dict

model = SITPCL_Entmax_Model(config, class_weights).to(config.device)
print(f'--> ✅ Model khởi tạo xong! Attention mode: {config.attention_mode} | Aux-relevance head: {config.use_aux_relevance_head}')

In [ ]:
# ==============================================================================
# STEP 6: OPTIMIZER & SCHEDULER
# ==============================================================================
optimizer = AdamW(model.parameters(), lr=config.learning_rate, weight_decay=config.weight_decay)
total_steps = (len(train_loader) // config.accum_steps) * config.num_epochs
warmup_steps = int(total_steps * config.warmup_ratio)
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=warmup_steps, num_training_steps=max(total_steps, 1))
scaler = torch.cuda.amp.GradScaler(enabled=config.use_amp)
print(f'--> ✅ Tổng số bước huấn luyện: {total_steps} (Warmup: {warmup_steps})')

In [ ]:
# ==============================================================================
# STEP 7: TRAINING LOOP (log thêm aux loss + alpha học được của attention)
# ==============================================================================
def evaluate_model(model, loader, target_names):
    model.eval()
    all_preds, all_labels, all_aux_preds, all_is_other = [], [], [], []
    cls_list = list(range(len(target_names)))
    with torch.no_grad():
        for batch in loader:
            input_ids = batch['input_ids'].to(config.device)
            attention_mask = batch['attention_mask'].to(config.device)
            mask_indices = batch['mask_indices'].to(config.device)
            labels = batch['labels'].to(config.device)
            with torch.cuda.amp.autocast(enabled=config.use_amp):
                _, logits, aux_logits, _ = model(
                    input_ids=input_ids, attention_mask=attention_mask, mask_indices=mask_indices,
                    dia_idx=batch['dia_idx'], speakers=batch['speakers'], targets=batch['targets'], labels=None
                )
            preds = torch.argmax(logits, dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.cpu().numpy())
            if aux_logits is not None:
                all_aux_preds.extend(torch.argmax(aux_logits, dim=1).cpu().numpy())
                all_is_other.extend((labels == 2).long().cpu().numpy())

    macro_f1 = f1_score(all_labels, all_preds, labels=cls_list, average='macro', zero_division=0)
    acc = accuracy_score(all_labels, all_preds)
    per_class_f1 = f1_score(all_labels, all_preds, labels=cls_list, average=None, zero_division=0)
    report = classification_report(all_labels, all_preds, labels=cls_list, target_names=target_names, zero_division=0, output_dict=True)
    aux_f1_other = f1_score(all_is_other, all_aux_preds, pos_label=1, average='binary', zero_division=0) if all_aux_preds else None
    return macro_f1, acc, per_class_f1, report, aux_f1_other

def print_learned_alphas(model):
    if config.attention_mode != 'entmax':
        return
    a_intra = model.sse.attention_intra.current_alpha().item()
    a_inter = model.sse.attention_inter.current_alpha().item()
    print(f'    [Alpha học được] intra-speaker={a_intra:.3f} | inter-speaker={a_inter:.3f}  (1.0=softmax mềm, 2.0=sparsemax cứng)')

best_dev_f1 = 0.0
patience_counter = 0
history = []

print('\n[*] BẮT ĐẦU TRAINING (BERTweet + Adaptive Entmax + Evidence + Aux-Relevance)...')
for epoch in range(1, config.num_epochs + 1):
    model.train()
    total_loss, total_focal, total_tpcl, total_fot, total_aux = 0.0, 0.0, 0.0, 0.0, 0.0
    optimizer.zero_grad()

    loop = tqdm(train_loader, desc=f'Epoch {epoch}/{config.num_epochs}')
    for step, batch in enumerate(loop):
        input_ids = batch['input_ids'].to(config.device)
        attention_mask = batch['attention_mask'].to(config.device)
        mask_indices = batch['mask_indices'].to(config.device)
        labels = batch['labels'].to(config.device)

        with torch.cuda.amp.autocast(enabled=config.use_amp):
            loss, _, _, l_dict = model(
                input_ids=input_ids, attention_mask=attention_mask, mask_indices=mask_indices,
                dia_idx=batch['dia_idx'], speakers=batch['speakers'], targets=batch['targets'], labels=labels
            )
            loss = loss / config.accum_steps

        scaler.scale(loss).backward()
        total_loss += loss.item() * config.accum_steps
        total_focal += l_dict.get('focal', 0.0)
        total_tpcl += l_dict.get('tpcl', 0.0)
        total_fot += l_dict.get('fot', 0.0)
        total_aux += l_dict.get('aux_relevance', 0.0)

        if (step + 1) % config.accum_steps == 0 or (step + 1) == len(train_loader):
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
            scheduler.step()

        loop.set_postfix({'loss': f"{total_loss/(step+1):.3f}", 'focal': f"{total_focal/(step+1):.3f}",
                           'tpcl': f"{total_tpcl/(step+1):.3f}", 'fot': f"{total_fot/(step+1):.3f}", 'aux': f"{total_aux/(step+1):.3f}"})

    avg_train_loss = total_loss / len(train_loader)
    dev_f1, dev_acc, dev_per_f1, _, dev_aux_f1 = evaluate_model(model, dev_loader, label_names)

    print(f'--> [Epoch {epoch}] Train Loss: {avg_train_loss:.4f} | Dev Macro-F1: {dev_f1*100:.2f}% | Dev Acc: {dev_acc*100:.2f}%' +
          (f' | Dev Aux-F1(Other,binary)={dev_aux_f1*100:.2f}%' if dev_aux_f1 is not None else ''))
    print_learned_alphas(model)
    history.append({'epoch': epoch, 'train_loss': avg_train_loss, 'dev_macro_f1': dev_f1, 'dev_acc': dev_acc})

    if dev_f1 > best_dev_f1:
        best_dev_f1 = dev_f1
        patience_counter = 0
        torch.save(model.state_dict(), config.best_model_path)
        print(f'    🏆 [Checkpoint tốt nhất] Dev F1: {best_dev_f1*100:.2f}%')
    else:
        patience_counter += 1
        print(f'    ⏳ [Không cải thiện] {patience_counter}/{config.early_stop_patience} epoch liên tiếp (Best: {best_dev_f1*100:.2f}%)')
        if patience_counter >= config.early_stop_patience:
            print(f'    🛑 [EARLY STOPPING] Dừng tại epoch {epoch} vì Dev Macro-F1 không cải thiện sau {config.early_stop_patience} epoch liên tiếp.')
            break

In [ ]:
# ==============================================================================
# STEP 8: ĐÁNH GIÁ TEST + BẢNG ĐỐI SÁNH VỚI 3 PHƯƠNG PHÁP CŨ
# ==============================================================================
print('\n' + '='*85)
print('[*] ĐÁNH GIÁ TRÊN TEST ZERO-SHOT VỚI CHECKPOINT TỐT NHẤT')
print('='*85)

if os.path.exists(config.best_model_path):
    model.load_state_dict(torch.load(config.best_model_path, map_location=config.device))
    print(f'--> [✓] Đã nạp checkpoint tốt nhất: {config.best_model_path}')

test_f1, test_acc, test_per_f1, test_report_dict, test_aux_f1 = evaluate_model(model, test_loader, label_names)
print_learned_alphas(model)

# Nhãn model phản ánh ĐÚNG config đã chạy — tránh gắn "+Evidence" khi flag đang tắt (so sánh công bằng).
model_name = f'SITPCL-FoT + {config.attention_mode.capitalize()}'
if config.use_aux_relevance_head:
    model_name += ' + AuxRelevance'
if config.use_evidence_injection:
    model_name += ' + Evidence [KHÔNG cùng input với 3 pp cũ!]'
model_name += ' (P1, Ours)'

res_dict = {
    'Model': model_name,
    'Mode': 'DEBUG' if config.debug_mode else 'FULL',
    'Same_input_as_3_pipeline_cu': not config.use_evidence_injection,
    'Accuracy': round(test_acc * 100, 2),
    'Macro_F1': round(test_f1 * 100, 2),
}
for idx, name in enumerate(label_names):
    if idx < len(test_per_f1):
        res_dict[f'F1_{name}'] = round(test_per_f1[idx] * 100, 2)
if test_aux_f1 is not None:
    res_dict['Aux_F1_Other_binary'] = round(test_aux_f1 * 100, 2)

results_df = pd.DataFrame([res_dict])
results_df.to_csv('stanceosaurus_p1_entmax_evidence_results.csv', index=False)

print('\n--- 📊 KẾT QUẢ P1 ---')
display(results_df)

if config.use_evidence_injection:
    print('\n⚠️ LƯU Ý: use_evidence_injection=True -> kết quả này KHÔNG dùng input giống 3 pipeline cũ.')
    print('   Không nên so sánh thẳng dòng này với bảng tham chiếu bên dưới. Hãy so sánh với kết quả')
    print('   chạy use_evidence_injection=False (cùng notebook) để biết riêng title đóng góp bao nhiêu.')
else:
    reference_rows = [
        {'Model': 'BERTweet baseline (Focal, EMNLP2022)', 'Macro_F1': 57.25, 'F1_Supporting': 56.05, 'F1_Refuting': 47.97, 'F1_Other': 67.71},
        {'Model': 'SITPCL (ACL2025 Findings)', 'Macro_F1': 43.14, 'F1_Supporting': 34.79, 'F1_Refuting': 34.49, 'F1_Other': 60.13},
        {'Model': 'SITPCL-FoT + Sparsemax (cứng)', 'Macro_F1': 36.98, 'F1_Supporting': 37.59, 'F1_Refuting': 32.91, 'F1_Other': 40.45},
    ]
    compare_df = pd.concat([pd.DataFrame(reference_rows), results_df], ignore_index=True)
    print('\n' + '='*85)
    print('🔥 BẢNG ĐỐI SÁNH CÔNG BẰNG (CÙNG INPUT train_conversations.json) VỚI 3 PHƯƠNG PHÁP ĐÃ CHẠY TRƯỚC ĐÓ:')
    print('='*85)
    display(compare_df)
    compare_df.to_csv('stanceosaurus_p1_final_comparison.csv', index=False)

report_df = pd.DataFrame(test_report_dict).T
print('\n--- Báo cáo chi tiết từng nhãn ---')
display(report_df.round(3))

if config.debug_mode:
    print('\n🎉 [XÁC NHẬN] DEBUG MODE CHẠY OK! Quay lại Cell 2 đổi DEBUG_MODE=False rồi Save & Run All.')
else:
    print('\n✅ [HOÀN TẤT] ĐÃ CHẠY XONG FULL DỮ LIỆU P1!')